### CASE TECNICO PYSPARK

In [1]:
##Import list
import pandas as pd
import matplotlib as plt
import os
import warnings
import logging
from pyspark.sql import SparkSession

# Suppress non-critical warnings
warnings.filterwarnings('ignore')
logging.getLogger("py4j").setLevel(logging.ERROR)

#set up global variables

CWD = os.getcwd()
CLIENTES_PATH = os.path.join(CWD, 'data/clients/data.json')
PEDIDOS_PATH = os.path.join(CWD, 'data/pedidos/data.json')

#start pyspark session

spark = (
    SparkSession.builder
    .appName("CaseTecnico")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.default.parallelism", "8")
    .getOrCreate()
)

# Set log level AFTER session creation
spark.sparkContext.setLogLevel("ERROR")



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 11:01:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, LongType, StringType, StructType, StructField
from pyspark.storagelevel import StorageLevel
from pyspark.sql.window import Window
from functools import reduce
from pyspark.sql.functions import broadcast

# explicit schemas avoid extra pass for inference
CLIENTES_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
])

PEDIDOS_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("client_id", LongType(), True),
    StructField("value", DecimalType(5, 2), True),
])


def load_data_from_json(path: str, schema: StructType, min_partitions: int | None = None) -> DataFrame:
    """Fast JSONL loader: schema-first, lazy, and optional repartition. Persists DataFrame in memory."""
    df = (
        spark.read
        .schema(schema)
        .option("multiLine", "false")
        .option("mode", "PERMISSIVE")
        .json(path)
    )

    if min_partitions is not None and df.rdd.getNumPartitions() < min_partitions:
        df = df.repartition(min_partitions)

    df = df.persist(StorageLevel.MEMORY_AND_DISK)
    return df



clientes_df = load_data_from_json(CLIENTES_PATH, CLIENTES_SCHEMA)
pedidos_df= load_data_from_json(PEDIDOS_PATH, PEDIDOS_SCHEMA, min_partitions=32)

print("clientes partitions:", clientes_df.rdd.getNumPartitions())
print("pedidos partitions:", pedidos_df.rdd.getNumPartitions())
print("schemas loaded successfully")

clientes partitions: 1


pedidos partitions: 32
schemas loaded successfully


## 1. Data Quality - Relatório de Falhas

In [ ]:
def regra_falha(df, motivo: str, ordem: int):
    return df.select(
        F.col("id"),
        F.lit(motivo).alias("motivo"),
        F.lit(ordem).alias("ordem_regra")
    )

# 1) Pedido sem valor
Sem_valor = regra_falha(
    pedidos_df.filter(F.col("value").isNull()),
    "pedido_sem_valor",
    1
)

# 2) ID de pedido duplicado
ids_duplicados_df = (
    pedidos_df
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
    .select("id")
)
ids_duplicados = regra_falha(ids_duplicados_df, "id_duplicado", 2)

# 3) Pedido com cliente inexistente (somente client_id válido)
cliente_inexistente = (
    pedidos_df.alias("p")
    .filter(F.col("p.client_id").isNotNull() & (F.col("p.client_id") < 0))
    .join(
        broadcast(clientes_df.select(F.col("id").alias("client_id_ref")).alias("c")),
        F.col("p.client_id") == F.col("c.client_id_ref"),
        "left_anti"
    )
    .select(F.col("p.id").alias("id"))
    .transform(lambda df: regra_falha(df, "cliente_inexistente", 3))
)

# 4) ID nulo
id_nullo = regra_falha(
    pedidos_df.filter(F.col("id").isNull()),
    "id_nulo",
    4
)

# 5) client_id nulo
client_id_nulo = regra_falha(
    pedidos_df.filter(F.col("client_id").isNull()),
    "client_id_nulo",
    5   
)

# 6) ID inválido (< 0)
id_invalido = regra_falha(
    pedidos_df.filter(F.col("id").isNotNull() & (F.col("id") < 0)),
    "id_invalido_menor_igual_zero",
    6
)

# 7) client_id inválido (<= 0)
client_id_invalido = regra_falha(
    pedidos_df.filter(F.col("client_id").isNotNull() & (F.col("client_id") < 0)),
    "client_id_invalido_menor_igual_zero",
    7
)

# 8) valor negativo ou zero
valro_negativo_ou_zero = regra_falha(
    pedidos_df.filter(F.col("value").isNotNull() & (F.col("value") <= 0)),
    "pedido_com_valor_negativo_ou_zero",
     8
)




# União de todas as regras
regras_falhas = [
    Sem_valor, ids_duplicados, cliente_inexistente, id_nullo, client_id_nulo,
    id_invalido, client_id_invalido,valro_negativo_ou_zero
]

falhas_df = (
    reduce(lambda acc, d: acc.unionByName(d), regras_falhas)
    .dropDuplicates(["id", "motivo"])
    .groupBy("id")
    .agg(
        F.concat_ws(", ", F.collect_list("motivo")).alias("motivo"),
        F.min("ordem_regra").alias("min_ordem")
    )
    .orderBy("min_ordem", "id")
    .select("id", "motivo")
)

# Saídas pedidas
falhas_df.show(100, truncate=False)

erros_por_categoria_df = (
    falhas_df
    .groupBy("motivo")
    .agg(F.count("*").alias("qtd_erros"))
    .orderBy(F.col("qtd_erros").desc(), F.col("motivo").asc())
)

erros_por_categoria_df.show(truncate=False)
falhas_df.agg(F.count("*").alias("total_erros")).show()


+------+-----------------------------------------------------------------+
|id    |motivo                                                           |
+------+-----------------------------------------------------------------+
|1534  |pedido_sem_valor, id_duplicado, pedido_com_valor_negativo_ou_zero|
|3502  |pedido_sem_valor, id_duplicado, pedido_com_valor_negativo_ou_zero|
|3679  |pedido_com_valor_negativo_ou_zero, pedido_sem_valor, id_duplicado|
|4322  |id_duplicado, pedido_sem_valor, pedido_com_valor_negativo_ou_zero|
|4724  |pedido_sem_valor, id_duplicado, pedido_com_valor_negativo_ou_zero|
|5774  |pedido_com_valor_negativo_ou_zero, pedido_sem_valor, id_duplicado|
|6322  |pedido_sem_valor, id_duplicado, pedido_com_valor_negativo_ou_zero|
|6622  |id_duplicado, pedido_sem_valor, pedido_com_valor_negativo_ou_zero|
|6884  |pedido_sem_valor, id_duplicado, pedido_com_valor_negativo_ou_zero|
|7116  |pedido_sem_valor, id_duplicado, pedido_com_valor_negativo_ou_zero|
|8597  |id_duplicado, ped

In [ ]:
# 8) Retorno sem pedido original correspondente
# Regra: para value < 0, deve existir (mesmo client_id, value positivo igual ao valor absoluto)
retornos_df = (
    pedidos_df
    .filter(F.col("value") < 0)
    .select(
        "id",
        "client_id",
        F.abs(F.col("value")).alias("valor_absoluto")
    )
)

pedidos_positivos_ref_df = (
    pedidos_df
    .filter(F.col("value") > 0)
    .select(
        F.col("client_id").alias("client_id_ref"),
        F.col("value").alias("valor_ref")
    )
    .distinct()
)

retornos_invalidos = (
    retornos_df.alias("r")
    .join(
        pedidos_positivos_ref_df.alias("p"),
        (F.col("r.client_id") == F.col("p.client_id_ref")) &
        (F.col("r.valor_absoluto") == F.col("p.valor_ref")),
        "left_anti"
    )
    .select(F.col("r.id").alias("id"))
    .transform(lambda df: regra_falha(df, "retorno_sem_pedido_original", 8))
)


In [33]:
# Pre-compute duplicated IDs efficiently
ids_duplicados_df = (
    pedidos_df
    .groupBy("id")
    .agg(F.count("*").alias("dup_count"))
    .filter(F.col("dup_count") > 1)
    .select(F.col("id").alias("dup_id"))
)

# Prepare client IDs as DataFrame (not Python list)
clientes_ids_df = clientes_df.select(F.col("id").alias("client_id_ref")).distinct()

# Single pass with joins instead of window/isin
pedidos_validos_df = (
    pedidos_df.alias("p")
    .select("id", "client_id", "value")
    # Anti-join to exclude duplicated IDs
    .join(broadcast(ids_duplicados_df), F.col("p.id") == F.col("dup_id"), "left_anti")
    # Left join to validate client exists
    .join(broadcast(clientes_ids_df), F.col("p.client_id") == F.col("client_id_ref"), "inner")
    .filter(
        F.col("value").isNotNull() 
        & (F.col("value") != 0) 
        & (F.col("value") > 0)
        & F.col("id").isNotNull() 
        & (F.col("id") >= 0)
        & F.col("client_id").isNotNull() 
        & (F.col("client_id") >= 0)
    )
    .select("p.id", "p.client_id", "p.value")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

pedidos_validos = pedidos_validos_df.count()
total_pedidos = pedidos_df.count()

print("Total pedidos:", total_pedidos)
print("Pedidos válidos:", pedidos_validos)

Total pedidos: 1100000
Pedidos válidos: 940493


In [34]:
# Optimized: aggregate first (reduces data size), then join with broadcast
cliente_totals_df = (
    pedidos_validos_df
    .groupBy("client_id")
    .agg(
        F.count("*").alias("qtd_pedidos"),
        F.sum("value").alias("valor_total"),
    )
    .join(
        broadcast(clientes_df.select(
            F.col("id").alias("client_id_ref"), 
            F.col("name").alias("client_name")
        )),
        F.col("client_id") == F.col("client_id_ref"),
        "inner"
    )
    .select(
        F.col("client_id").alias("id_cliente"),
        F.col("client_name").alias("nome_cliente"),
        F.col("qtd_pedidos"),
        F.col("valor_total")
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

cliente_totals_df.show(50, truncate=False)

+----------+------------------+-----------+-----------+
|id_cliente|nome_cliente      |qtd_pedidos|valor_total|
+----------+------------------+-----------+-----------+
|123456    |Inês Siqueira     |469734     |23698016.90|
|9047      |Zachary Reis      |61         |4002.08    |
|4494      |Vitor Marques     |68         |3937.06    |
|2695      |Wanda Silva       |61         |3736.53    |
|2756      |Inês Siqueira     |60         |3735.58    |
|8566      |Tereza Leal       |66         |3731.10    |
|6135      |Mariana Melo      |71         |3700.71    |
|5221      |Yasmin Carvalho   |65         |3679.35    |
|9266      |Tereza Leal       |67         |3678.17    |
|2379      |Gustavo Pontes    |65         |3657.84    |
|8317      |Sofia Castro      |66         |3656.86    |
|7543      |Vitória Andrade   |69         |3656.43    |
|849       |Breno Soares      |66         |3651.20    |
|6532      |João Batista      |68         |3650.20    |
|857       |Julio Viana       |62         |3645.

In [35]:
# Optimized: compute statistics in minimal passes (2 actions instead of 3)
stats = cliente_totals_df.agg(F.mean("valor_total").alias("media")).collect()[0]
media = stats["media"]

# Compute all quantiles in a single pass
percentil_10, mediana, percentil_90 = cliente_totals_df.approxQuantile("valor_total", [0.1, 0.5, 0.9], 0.01)

print(f"Valor médio total por cliente: {media:.2f}")
print(f"Mediana do valor total por cliente: {mediana:.2f}")
print(f"10º percentil do valor total por cliente: {percentil_10:.2f}")
print(f"90º percentil do valor total por cliente: {percentil_90:.2f}")

Valor médio total por cliente: 4746.44
Mediana do valor total por cliente: 2367.82
10º percentil do valor total por cliente: 1878.04
90º percentil do valor total por cliente: 2893.59


In [36]:
clientes_acima_media_df = (
    cliente_totals_df
    .filter(F.col("valor_total") > media)
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_acima_media_df.show(50, truncate=False)

+----------+-------------+-----------+-----------+
|id_cliente|nome_cliente |qtd_pedidos|valor_total|
+----------+-------------+-----------+-----------+
|123456    |Inês Siqueira|469734     |23698016.90|
+----------+-------------+-----------+-----------+



In [37]:
clientes_media_truncada_df = (
    cliente_totals_df
    .filter(
        (F.col("valor_total") >= percentil_10) &
        (F.col("valor_total") <= percentil_90)
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_media_truncada_df.show(50, truncate=False)

+----------+------------------+-----------+-----------+
|id_cliente|nome_cliente      |qtd_pedidos|valor_total|
+----------+------------------+-----------+-----------+
|1018      |Thiago Araújo     |60         |2893.59    |
|888       |Priscila Machado  |62         |2893.50    |
|6224      |Bernardo Pinto    |50         |2893.37    |
|3470      |Xuxa Moura        |63         |2892.88    |
|8925      |Camila Freitas    |50         |2892.66    |
|6920      |Wagner Teixeira   |60         |2892.53    |
|8096      |Ximena Costa      |55         |2892.27    |
|9652      |Elaine Coelho     |55         |2892.17    |
|1671      |Yuri Neves        |57         |2892.04    |
|5529      |Giovanna Ramos    |57         |2892.03    |
|5741      |Tatiana Borges    |53         |2891.92    |
|1990      |Roberta Campos    |56         |2891.62    |
|4929      |Giovanna Ramos    |55         |2891.12    |
|8039      |Rafaela Vieira    |52         |2891.00    |
|5284      |Letícia Porto     |57         |2890.

##CASO ID 123456

os pedidos do id 123456, estao com o valor muito elevado e pode possivelmente ser um erro da base de dados, logo, vou explorar os valores relacionados a esse ID e executar os calculos e comparar os resultados.

In [38]:
pedidos_ines = (
    pedidos_validos_df.alias("pedidos")
    .filter(F.col("client_id") == 123456)
    .orderBy(F.col("value").desc())
    .select("id", "value")
)

pedidos_ines.show(10,truncate=False)

quantidade_ines99 = pedidos_ines.filter(F.col("value") == 99.99).count()

print("Quantidade de pedidos de Inês com valor 99.99:", quantidade_ines99)

Ines_valores_repitidos_df = (
    pedidos_ines.groupBy("value")
    .agg(F.count("*").alias("qtd_repeticoes"))
    .filter(F.col("qtd_repeticoes") > 1)
    .orderBy(F.col("qtd_repeticoes").desc(), F.col("value").asc())
)

Ines_valores_repitidos_df.show(10, truncate=False)


+--------+-----+
|id      |value|
+--------+-----+
|96638460|99.99|
|33658689|99.99|
|50816623|99.99|
|1230531 |99.99|
|29038115|99.99|
|63255621|99.99|
|64450996|99.99|
|85827365|99.99|
|58270999|99.99|
|56757517|99.99|
+--------+-----+
only showing top 10 rows
Quantidade de pedidos de Inês com valor 99.99: 26
+-----+--------------+
|value|qtd_repeticoes|
+-----+--------------+
|99.72|75            |
|51.41|74            |
|59.25|74            |
|42.32|71            |
|51.94|71            |
|63.11|71            |
|66.57|71            |
|83.85|71            |
|84.41|71            |
|16.36|70            |
+-----+--------------+
only showing top 10 rows


In [39]:
cliente_totals_df.filter(F.col("id_cliente") != 123456).show(truncate=False)   

+----------+---------------+-----------+-----------+
|id_cliente|nome_cliente   |qtd_pedidos|valor_total|
+----------+---------------+-----------+-----------+
|9047      |Zachary Reis   |61         |4002.08    |
|4494      |Vitor Marques  |68         |3937.06    |
|2695      |Wanda Silva    |61         |3736.53    |
|2756      |Inês Siqueira  |60         |3735.58    |
|8566      |Tereza Leal    |66         |3731.10    |
|6135      |Mariana Melo   |71         |3700.71    |
|5221      |Yasmin Carvalho|65         |3679.35    |
|9266      |Tereza Leal    |67         |3678.17    |
|2379      |Gustavo Pontes |65         |3657.84    |
|8317      |Sofia Castro   |66         |3656.86    |
|7543      |Vitória Andrade|69         |3656.43    |
|849       |Breno Soares   |66         |3651.20    |
|6532      |João Batista   |68         |3650.20    |
|857       |Julio Viana    |62         |3645.72    |
|9147      |Zachary Reis   |59         |3645.27    |
|2781      |Ivan Almeida   |73         |3641.0

In [30]:
media_A = cliente_totals_df.filter(F.col("id_cliente") != 123456).agg(F.mean("valor_total").alias("media_valor_total")).collect()[0]["media_valor_total"]

print(f"Valor médio total por cliente: {media_A:.2f}")

mediana_A = cliente_totals_df.filter(F.col("id_cliente") != 123456).approxQuantile("valor_total", [0.5], 0.01)[0]

print(f"Mediana do valor total por cliente: {mediana_A:.2f}")

percentil_10_A, percentil_90_A = cliente_totals_df.filter(F.col("id_cliente") != 123456).approxQuantile("valor_total", [0.1, 0.9], 0.01)

print(f"10º percentil do valor total por cliente: {percentil_10_A:.2f}")

print(f"90º percentil do valor total por cliente: {percentil_90_A:.2f}")

Valor médio total por cliente: 2377.11
Mediana do valor total por cliente: 2363.45
10º percentil do valor total por cliente: 1872.75
90º percentil do valor total por cliente: 2883.37


In [31]:
clientes_acima_media_df_A = (
    cliente_totals_df
    .filter(
        (F.col("id_cliente") != 123456) & (F.col("valor_total") > media_A)
        )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_acima_media_df_A.show(50, truncate=False)

+----------+------------------+-----------+-----------+
|id_cliente|nome_cliente      |qtd_pedidos|valor_total|
+----------+------------------+-----------+-----------+
|9047      |Zachary Reis      |61         |4002.08    |
|4494      |Vitor Marques     |68         |3937.06    |
|2695      |Wanda Silva       |61         |3736.53    |
|2756      |Inês Siqueira     |60         |3735.58    |
|8566      |Tereza Leal       |66         |3731.10    |
|6135      |Mariana Melo      |71         |3700.71    |
|5221      |Yasmin Carvalho   |65         |3679.35    |
|9266      |Tereza Leal       |67         |3678.17    |
|2379      |Gustavo Pontes    |65         |3657.84    |
|8317      |Sofia Castro      |66         |3656.86    |
|7543      |Vitória Andrade   |69         |3656.43    |
|849       |Breno Soares      |66         |3651.20    |
|6532      |João Batista      |68         |3650.20    |
|857       |Julio Viana       |62         |3645.72    |
|9147      |Zachary Reis      |59         |3645.

In [32]:
clientes_media_truncada_df_A = (
    cliente_totals_df
    .filter(
        (F.col("id_cliente") != 123456) &
        (F.col("valor_total") >= percentil_10_A) &
        (F.col("valor_total") <= percentil_90_A)
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_media_truncada_df.show(50, truncate=False)

+----------+------------------+-----------+-----------+
|id_cliente|nome_cliente      |qtd_pedidos|valor_total|
+----------+------------------+-----------+-----------+
|1018      |Thiago Araújo     |60         |2893.59    |
|888       |Priscila Machado  |62         |2893.50    |
|6224      |Bernardo Pinto    |50         |2893.37    |
|3470      |Xuxa Moura        |63         |2892.88    |
|8925      |Camila Freitas    |50         |2892.66    |
|6920      |Wagner Teixeira   |60         |2892.53    |
|8096      |Ximena Costa      |55         |2892.27    |
|9652      |Elaine Coelho     |55         |2892.17    |
|1671      |Yuri Neves        |57         |2892.04    |
|5529      |Giovanna Ramos    |57         |2892.03    |
|5741      |Tatiana Borges    |53         |2891.92    |
|1990      |Roberta Campos    |56         |2891.62    |
|4929      |Giovanna Ramos    |55         |2891.12    |
|8039      |Rafaela Vieira    |52         |2891.00    |
|5284      |Letícia Porto     |57         |2890.